In [31]:
import pandas as pd
import numpy as np
import re
import pdb
from collections import defaultdict
import os

In [32]:
# Einlesen der Date aus dem Notebook "1_PuRe Rohdatenextrakt.jpynb"
df_pure = pd.read_excel('Daten/df_pure.xlsx')

# Alternativ kann ich auch die Daten aus dem json-download über Openrefine einlesen
#df_pure = pd.read_excel('download-json.xlsx')

# Zur Sicherheit die Anzahl an IDs = Datensätzen zählen
df_pure['90999_ItemID'].nunique() 

447

In [33]:
df_pure

,90999_ItemID,data - versionNumber,data - modificationDate,data - versionState,data - versionPid,data - lastModificationDate,data - publicState,data - objectPid,data - creationDate,data - message,...,data - latestVersion - objectId,data - latestVersion - versionNumber,data - latestVersion - modificationDate,data - latestVersion - versionState,data - latestVersion - versionPid,data - latestVersion - modifier - objectId,Item_seq,Name,kleinster_wert,1103_Ersterscheinungsdatum
0,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-BC65-7,2026-03-25T10:54:08.925+00:00,NaN,...,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,user_1241569,1,Luca Cigna,2026.0,2026.0
1,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
2,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
3,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,Nina Lopez-Uroz,NaN,NaN
4,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-5C0E-0,2022-11-02T13:32:56.911+00:00,NaN,...,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,user_1241569,1,Vincenzo Maccarrone,2022.0,2022.0
2333,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,Arianna Tassinari,NaN,NaN
2334,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
2335,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN


In [34]:
# Umbenennung der hier relevanten Spalten
df_pure = df_pure.rename(columns={
    'data - metadata - title':'1001_Titel',
    'data - metadata - genre':'1002_Genre',
    'data - metadata - degree' : '1021_Degree',
    'data - metadata - sources - _ - title':'1101_Quellentitel',
    'data - metadata - reviewMethod':'1108_Review',
    '1103_Ersterscheinungsdatum':'Datum',
    'data - files - _ - content':'4_File-content',
    'data - files - _ - metadata - title':'2_File-title',
    'data - files - _ - visibility':'5_File-visibility',
    'data - files - _ - storage':'7_File-Storage',
    'data - files - _ - metadata - contentCategory':'3_File-Category',
    'data - files - _ - metadata - description':'1_File-Description',
    'data - files - _ - metadata - oaStatus':'6_File-OA',
    'data - files - _ - metadata - license':'8_File-Licence',
    'data - files - _ - metadata - embargoUntil':'9_File-Embargo',
    'data - objectId':'90999_ItemID',}) 

df_pure



,90999_ItemID,data - versionNumber,data - modificationDate,data - versionState,data - versionPid,data - lastModificationDate,data - publicState,data - objectPid,data - creationDate,data - message,...,data - latestVersion - objectId,data - latestVersion - versionNumber,data - latestVersion - modificationDate,data - latestVersion - versionState,data - latestVersion - versionPid,data - latestVersion - modifier - objectId,Item_seq,Name,kleinster_wert,Datum
0,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-BC65-7,2026-03-25T10:54:08.925+00:00,NaN,...,item_3699285,2.0,2026-04-23T12:29:32.939+00:00,RELEASED,hdl:21.11116/0000-0012-FA72-2,user_1241569,1,Luca Cigna,2026.0,2026.0
1,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
2,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
3,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,Nina Lopez-Uroz,NaN,NaN
4,item_3699285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-5C0E-0,2022-11-02T13:32:56.911+00:00,NaN,...,item_3458140,2.0,2022-11-09T13:27:47.664+00:00,RELEASED,hdl:21.11116/0000-000B-6898-5,user_1241569,1,Vincenzo Maccarrone,2022.0,2022.0
2333,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,Arianna Tassinari,NaN,NaN
2334,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
2335,item_3458140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN


In [35]:
# Auswahl der Spalten und Übertrag in ein neues df

cropped_df = df_pure[['1001_Titel',
                      '1002_Genre',
                      'Datum',
                      '1101_Quellentitel',
                      '1_File-Description',
                      '2_File-title',
                      '3_File-Category',
                      '4_File-content',
                      '5_File-visibility',
                      '6_File-OA', 
                      '7_File-Storage',
                      '8_File-Licence',
                      '9_File-Embargo',
                      '90999_ItemID']]
# cropped_df

In [36]:
# Datumsspalte bereinigen, da durch Excel-Export verändert und Werte auffüllen
cropped_df['Datum'] = cropped_df['Datum'].astype(str).str.extract(r'(\d{4})')
cropped_df['Datum'] = cropped_df['Datum'].ffill() 

# Spalteninhalt wieder als Zahl definieren
cropped_df['Datum'] = cropped_df['Datum'].astype(int)

# Blick auf die enthaltenen Daten 
cropped_df['Datum'].unique()

C:\Users\cgmol\AppData\Local\Temp\ipykernel_71500\475377001.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cropped_df['Datum'] = cropped_df['Datum'].astype(str).str.extract(r'(\d{4})')
C:\Users\cgmol\AppData\Local\Temp\ipykernel_71500\475377001.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cropped_df['Datum'] = cropped_df['Datum'].ffill()
C:\Users\cgmol\AppData\Local\Temp\ipykernel_71500\475377001.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Tr

array([2026, 2025, 2024, 2023, 2017, 2022, 2020, 2021])

In [37]:
cropped_df

,1001_Titel,1002_Genre,Datum,1101_Quellentitel,1_File-Description,2_File-title,3_File-Category,4_File-content,5_File-visibility,6_File-OA,7_File-Storage,8_File-Licence,9_File-Embargo,90999_ItemID
0,The Limits of Green Concertation: Business Fra...,ARTICLE,2026,New Political Economy,Full text,mpifg_zs26_1803.pdf,any-fulltext,/rest/items/item_3699285_2/component/file_3699...,AUDIENCE,NaN,INTERNAL_MANAGED,https://creativecommons.org/licenses/by/4.0/,NaN,item_3699285
1,NaN,ARTICLE,2026,NaN,Full text open access via publisher,https://doi.org/10.1080/13563467.2026.2643433,publisher-version,https://doi.org/10.1080/13563467.2026.2643433,PUBLIC,HYBRID,EXTERNAL_URL,NaN,NaN,item_3699285
2,NaN,ARTICLE,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,item_3699285
3,NaN,ARTICLE,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,item_3699285
4,NaN,ARTICLE,2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,item_3699285
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332,Worker Solidarity among Gig and Precarious Wor...,CONTRIBUTION_TO_HANDBOOK,2022,The Routledge Handbook of the Gig Economy,Full text,mpifg_am23_193.pdf,any-fulltext,/rest/items/item_3458140_2/component/file_3473...,AUDIENCE,NaN,INTERNAL_MANAGED,NaN,NaN,item_3458140
2333,NaN,CONTRIBUTION_TO_HANDBOOK,2022,NaN,Full text via publisher,https://doi.org/10.4324/9781003161875-16,publisher-version,https://doi.org/10.4324/9781003161875-16,PUBLIC,NOT_SPECIFIED,EXTERNAL_URL,NaN,NaN,item_3458140
2334,NaN,CONTRIBUTION_TO_HANDBOOK,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,item_3458140
2335,NaN,CONTRIBUTION_TO_HANDBOOK,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,item_3458140


# Hier die gewünschten Jahre für Abfrage eintragen

In [69]:
years = [2023, 2024, 2025]   # oder bei einem Einzahljahr so [2025]

df_matching_years = cropped_df[cropped_df['Datum'].isin(years)].copy()

In [70]:
# Nochmal kurz Nachzählen, dass uns keine IDs abhanden gekommen sind
def count_unique_itemids(df, col='90999_ItemID'):
    return df[col].nunique()

print("Datensätze in der Datei nach Bereinigung:", count_unique_itemids(cropped_df))

print("Datensätze in der Eingabedatei:", count_unique_itemids(df_pure))

print("Datensätze mit passenden Erscheinungsjahren:", count_unique_itemids(df_matching_years))

df_matching_years.to_excel('Rohdaten_OA.xlsx', index=False)


Datensätze in der Datei nach Bereinigung: 447
Datensätze in der Eingabedatei: 447
Datensätze mit passenden Erscheinungsjahren: 399


In [46]:
# Schritt der Reduzierung auf bestimmte Positionen im Status - da wir beschlossen haben Emeriti, Associates, Visiting und Other rauszulassen ???

#desired_statuses = ['Doctoral Student','Postdoc', 'Senior Researcher','Research Group Leader','Director']
#df_final = df[df['Status'].isin(desired_statuses)]

Wie viele Datensätze befinden sich im Embargo und bis zu welchem Datum?

In [71]:
embargo_counts= df_matching_years['9_File-Embargo'].value_counts()

# Sortiere aufsteigend nach Datum - hier können bestimmte Titel mitgerechnet werden, wenn Embargo vor Zahlenverwertung aufgehoben sein wird
embargo_counts_sorted_asc = embargo_counts.sort_index(ascending=True)


embargo_counts_sorted_asc

9_File-Embargo
2026-04-30    4
2026-06-30    1
2026-07-31    1
2026-08-31    2
2026-09-30    2
2026-10-31    1
2026-12-31    1
2027-02-28    1
Name: count, dtype: int64

In [72]:
desired_statuses = ['Full text open access','Full text open access via publisher']

df = df_matching_years[df_matching_years['1_File-Description'].isin(desired_statuses)]

In [73]:
df

,1001_Titel,1002_Genre,Datum,1101_Quellentitel,1_File-Description,2_File-title,3_File-Category,4_File-content,5_File-visibility,6_File-OA,7_File-Storage,8_File-Licence,9_File-Embargo,90999_ItemID
18,From Adorno to 50 Cent: Financialized Platform...,ARTICLE,2025,Finance and Society,Full text open access,FS_12_2026_Kullick.pdf,any-fulltext,/rest/items/item_3674340_2/component/file_3705...,PUBLIC,GREEN,INTERNAL_MANAGED,https://creativecommons.org/licenses/by-sa/4.0/,NaN,item_3674340
19,NaN,ARTICLE,2025,NaN,Full text open access via publisher,https://doi.org/10.1017/fas.2025.10015,publisher-version,https://doi.org/10.1017/fas.2025.10015,PUBLIC,GOLD,EXTERNAL_URL,NaN,NaN,item_3674340
24,The Comparative Political Economy of the Green...,ARTICLE,2025,Regulation & Governance,Full text open access,RG_20_2026_Cigna.pdf,any-fulltext,/rest/items/item_3674550_3/component/file_3704...,PUBLIC,GREEN,INTERNAL_MANAGED,https://creativecommons.org/licenses/by/4.0/,NaN,item_3674550
25,NaN,ARTICLE,2025,NaN,Full text open access via publisher,https://doi.org/10.1111/rego.70080,publisher-version,https://doi.org/10.1111/rego.70080,PUBLIC,HYBRID,EXTERNAL_URL,NaN,NaN,item_3674550
44,Overextended: The European Disunion at a Cross...,ARTICLE,2025,American Affairs,Full text open access,AmerA_9_2025_Streeck.pdf,any-fulltext,/rest/items/item_3637800_5/component/file_3702...,PUBLIC,GREEN,INTERNAL_MANAGED,NaN,NaN,item_3637800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2161,NaN,ARTICLE,2023,NaN,Full text open access via publisher,https://doi.org/10.1080/13501763.2023.2202684,publisher-version,https://doi.org/10.1080/13501763.2023.2202684,PUBLIC,HYBRID,EXTERNAL_URL,NaN,NaN,item_3509134
2165,Top Wealth and Its Historical Origins: An Anal...,PAPER,2023,MPIfG Discussion Paper,Full text open access,mpifg_dp23_1.pdf,any-fulltext,/rest/items/item_3506276_3/component/file_3506...,PUBLIC,NOT_SPECIFIED,INTERNAL_MANAGED,NaN,NaN,item_3506276
2177,What Growth Strategies Do Citizens Want? Evide...,PAPER,2023,MPIfG Discussion Paper,Full text open access,mpifg_dp23_4.pdf,any-fulltext,/rest/items/item_3528109_3/component/file_3528...,PUBLIC,NOT_SPECIFIED,INTERNAL_MANAGED,NaN,NaN,item_3528109
2231,Europa-Kolumne: Die Europäische Zentralbank im...,ARTICLE,2023,Merkur,Full text open access,Merkur_889_2023_Höpner.pdf,any-fulltext,/rest/items/item_3515352_2/component/file_3515...,PUBLIC,NOT_SPECIFIED,INTERNAL_MANAGED,NaN,NaN,item_3515352


In [ ]:
print("Datensätze in der Datei mit :", count_unique_itemids(cropped_df))
print("Datensätze davon mit Full text open access (via publisher):", count_unique_itemids(df))


Datensätze in der Datei mit : 447
Datensätze mit Full text open access (via publisher): 224


In [77]:
# Extrahiere das Jahr aus dem Datum (angenommen, Datum ist im Format 'YYYY-MM-DD' oder ähnlich)
df['Jahr'] = pd.to_datetime(df['Datum']).dt.year

# Gruppiere nach Jahr und 0999_ItemID, um für jedes Item nur einen Eintrag zu haben
# Priorität: zuerst "Full text open access", dann "Full text open access via publisher"
# Wir verwenden .first() nach Sortierung, um die Priorität zu gewährleisten

# Sortiere nach Priorität: "Full text open access" zuerst, dann "Full text open access via publisher"
# Wir setzen die Reihenfolge der Werte so, dass "Full text open access" vor "Full text open access via publisher" kommt
df_sorted = df.sort_values(
    by=['Jahr', '0999_ItemID', '1_File-Description'],
    key=lambda x: x.map({
        'Full text open access': 0,
        'Full text open access via publisher': 1,
        # Alle anderen Werte werden als 2 behandelt (höhere Priorität = niedrigere Zahl)
    }).fillna(2)
)

# Entferne Duplikate pro 0999_ItemID, behalte nur den ersten Eintrag (mit Priorität)
#df_unique = df_sorted.drop_duplicates(subset=['Jahr', '0999_ItemID'], keep='first')

# Zähle nun pro Jahr die Anzahl der verschiedenen Typen
result1 = df_sorted.groupby(['Jahr', '1_File-Description']).size().unstack(fill_value=0)

# Benenne die Spalten für bessere Lesbarkeit
result1.columns.name = None
result1 = result1.rename(columns={
    'Full text open access': 'Full text open access',
    'Full text open access via publisher': 'Full text open access via publisher'
})

print("1. Anzahl pro Jahr und Open Access Typ (je Item nur einmal):")
result1




C:\Users\cgmol\AppData\Local\Temp\ipykernel_71500\2803828523.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Jahr'] = pd.to_datetime(df['Datum']).dt.year


KeyError: '0999_ItemID'

In [81]:
#df['Jahr'] = pd.to_datetime(df['Datum']).dt.year

df_sorted2 = df.sort_values(
    by=['90999_ItemID', '6_File-OA'],
    key=lambda x: x.map({
        'GOLD': 0,
        'GREEN': 1,
        'HYBRID': 2,
        # Alle anderen Werte werden als 3 behandelt (höhere Priorität = niedrigere Zahl)
    }).fillna(3)
)

# Entferne Duplikate pro 0999_ItemID, behalte nur den ersten Eintrag (mit Priorität)
#df_unique2 = df_sorted2.drop_duplicates(subset=['Jahr', '0999_ItemID'], keep='first')

# Zähle nun pro Jahr die Anzahl der verschiedenen Typen
result2 = df_sorted2.groupby(['Datum', '6_File-OA']).size().unstack(fill_value=0)

# Benenne die Spalten für bessere Lesbarkeit
result2.columns.name = None


result2


,GOLD,GREEN,HYBRID,MISCELLANEOUS,NOT_SPECIFIED
Datum,,,,,
2023,8,41,22,3,14
2024,11,55,38,8,12
2025,34,49,30,8,2


In [ ]:
df_sorted2

,4_File-content,1_File-Description,2_File-title,3_File-Category,5_File-visibility,6_File-OA,7_File-Storage,8_File-Licence,9_File-Embargo,0102_Erscheinungsdatum,0103_OnlineDatum,0999_ItemID,kleinster_wert,Datum,Jahr
19,https://doi.org/10.14746/sr.2023.7.2.01,Full text open access via publisher,https://doi.org/10.14746/sr.2023.7.2.01,publisher-version,PUBLIC,GOLD,EXTERNAL_URL,NaN,NaN,2023,2023,item_3508417,2023.0,2023,2023
58,https://doi.org/10.17645/pag.11267,Full text open access via publisher,https://doi.org/10.17645/pag.11267,publisher-version,PUBLIC,GOLD,EXTERNAL_URL,NaN,NaN,2024,2026,item_3696684,2024.0,2024,2024
233,/rest/items/item_3688842_3/component/file_3689...,Full text open access,mpifg_dp25_7.pdf,any-fulltext,PUBLIC,GOLD,INTERNAL_MANAGED,https://creativecommons.org/licenses/by/4.0/de...,NaN,2025,2023,item_3688842,2023.0,2023,2023
261,https://econsoc.mpifg.de/51140/05_Ergen_Econso...,Full text open access via publisher,https://econsoc.mpifg.de/51140/05_Ergen_Econso...,publisher-version,PUBLIC,GOLD,EXTERNAL_URL,NaN,NaN,2025,2025,item_3685535,2025.0,2025,2025
282,https://doi.org/10.1177/23328584251400269,Full text open access via publisher,https://doi.org/10.1177/23328584251400269,publisher-version,PUBLIC,GOLD,EXTERNAL_URL,NaN,NaN,2025,2025,item_3683852,2025.0,2025,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2159,/rest/items/item_3528109_3/component/file_3528...,Full text open access,mpifg_dp23_4.pdf,any-fulltext,PUBLIC,NOT_SPECIFIED,INTERNAL_MANAGED,NaN,NaN,2023,2023,item_3528109,2023.0,2023,2023
2213,/rest/items/item_3515352_2/component/file_3515...,Full text open access,Merkur_889_2023_Höpner.pdf,any-fulltext,PUBLIC,NOT_SPECIFIED,INTERNAL_MANAGED,NaN,NaN,2023,2022,item_3515352,2022.0,2022,2022
2230,/rest/items/item_3511109_1/component/file_3511...,Full text open access,mpifg_diss22_Sonntag.pdf,any-fulltext,PUBLIC,NOT_SPECIFIED,INTERNAL_MANAGED,NaN,NaN,2022,2023,item_3511109,2022.0,2022,2022
2259,/rest/items/item_3508002_1/component/file_3508...,Full text open access,mpifg_dp23_2.pdf,any-fulltext,PUBLIC,NOT_SPECIFIED,INTERNAL_MANAGED,NaN,NaN,2023,2023,item_3508002,2023.0,2023,2023


In [ ]:
df['Jahr'] = pd.to_datetime(df['Datum']).dt.year


# Entferne Duplikate pro 0999_ItemID, behalte nur den ersten Eintrag (mit Priorität)
df_unique2 = df_sorted2.drop_duplicates(subset=['Jahr', '0999_ItemID'], keep='first')

# Zähle nun pro Jahr die Anzahl der verschiedenen Typen
result2 = df_sorted2.groupby(['Jahr', '6_File-OA']).size().unstack(fill_value=0)

# Benenne die Spalten für bessere Lesbarkeit
result2.columns.name = None


result2


In [83]:
# 2. Zähle pro Jahr und 0999_ItemID: Anzahl der Einträge für "HYBRID", "GREEN", "GOLD"
# Annahme: Die Spalte "5_File-Visibility" enthält die Werte "HYBRID", "GREEN", "GOLD" oder andere
# Wir filtern nur die gewünschten Werte

# Filtere nur die Zeilen mit den gewünschten Werten
df_filtered = df[df['6_File-OA'].isin(['HYBRID', 'GREEN', 'GOLD'])]

# Gruppiere nach Jahr und 0999_ItemID und zähle die Einträge
result2 = df_filtered.groupby(['Datum', '90999_ItemID'])['6_File-OA'].count().unstack(fill_value=0)

# Benenne die Spalten
result2.columns.name = None
result2 = result2.rename(columns={
    'HYBRID': 'HYBRID',
    'GREEN': 'GREEN',
    'GOLD': 'GOLD'
})

print("\n2. Anzahl pro Jahr und 90999_ItemID für HYBRID, GREEN, GOLD:")
result2


2. Anzahl pro Jahr und 90999_ItemID für HYBRID, GREEN, GOLD:


,item_3489209,item_3490194,item_3492350,item_3493346,item_3496301,item_3501133,item_3505953,item_3508417,item_3509134,item_3510300,...,item_3691588,item_3692950,item_3693118,item_3693143,item_3693194,item_3693350,item_3693451,item_3695165,item_3695172,item_3695187
Datum,,,,,,,,,,,,,,,,,,,,,
2023,2,2,1,1,1,2,2,2,1,2,...,0,0,0,0,0,0,0,0,0,0
2024,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,2,0,1,2,1
2025,0,0,0,0,0,0,0,0,0,0,...,2,2,2,1,1,0,2,0,0,0
